In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from spectral_detection.analysis.pca_pipeline import pipeline

We first load the pipeline that we will be using

In [2]:
pipe = pipeline(L=21, H=32, K=10)

Let us test on a couple of individual data sets first.

In [4]:
X1, ids1 = pipe.load_pt(r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt")

X1_pca = pipe.fit_transform(X1)

print("Original shape:", X1.shape)
print("Reduced shape:", X1_pca.shape)

Original shape: (817, 6720)
Reduced shape: (817, 335)


In [5]:
X2, ids2 = pipe.load_pt(r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt")

X2_pca = pipe.transform(X2)

print("Original shape:", X2.shape)
print("Reduced shape:", X2_pca.shape)

Original shape: (2000, 6720)
Reduced shape: (2000, 335)


We can now merge all the datasets and do PCA on the whole data.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [7]:
# -----------------------------
#  List the 4 .pt files
# -----------------------------
pt_paths = [
    r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/triviaQA_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/mmlu_Judgelabels_and_eigs_top10.pt",
    r"../data/spectral/math_Judgelabels_and_eigs_top10.pt"
]

# Optional: dataset names for tracking
dataset_names = ["truthfulqa", "triviaqa", "mmlu", "math"]

# -----------------------------
# Load + featurize all four, then stack
# -----------------------------
X_all = []
ids_all = []
source_all = []

for path, ds_name in zip(pt_paths, dataset_names):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    X, ids = pipe.load_pt(path, feature_key="eig_top10")  # X shape: [N, L*K] after head-avg + signed-log
    X_all.append(X)
    ids_all.extend(ids)
    source_all.extend([ds_name] * len(ids))

X_all = np.vstack(X_all)  # shape: [N_total, L*K]
print("Combined X shape:", X_all.shape)

Combined X shape: (6817, 6720)


In [8]:
# -----------------------------
# 3) Fit PCA on the combined dataset
# -----------------------------
X_pca = pipe.fit_transform(X_all)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (6817, 1393)
Num PCs kept: 1393
Explained variance retained: 0.9500064


In [9]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)
df_pca.insert(0, "id", ids_all)
df_pca.insert(1, "dataset", source_all)

df_pca.head()

,id,dataset,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,...,PC1384,PC1385,PC1386,PC1387,PC1388,PC1389,PC1390,PC1391,PC1392,PC1393
0,truthfulqa_00000_t0.1_ans00,truthfulqa,-30.033062,30.513218,28.804989,-24.437023,-18.623959,-9.327746,-1.688647,-11.716867,...,-0.240237,-0.495906,-0.326636,0.089442,-0.281747,0.417938,0.419732,-0.344318,-0.257110,0.243547
1,truthfulqa_00001_t0.1_ans00,truthfulqa,-41.503330,9.963426,-12.280369,23.449280,-3.379167,5.100118,16.664856,7.091613,...,-0.023484,-0.212478,-0.494584,-0.513165,-0.316231,-0.542960,0.088251,0.281707,-0.000039,0.331964
2,truthfulqa_00002_t0.1_ans00,truthfulqa,-40.586487,14.426265,11.908709,-10.016600,9.774280,-8.877830,-8.240797,-0.351241,...,-0.749912,0.491576,0.083951,0.747851,-0.421894,0.788515,-0.788881,0.415017,-0.129002,-0.164531
3,truthfulqa_00003_t0.1_ans00,truthfulqa,-44.032581,8.705839,-7.369984,-4.381597,6.520583,1.212528,-7.974981,9.117405,...,0.386629,0.342452,-0.128083,0.370429,-0.172539,-0.451733,0.808168,0.175775,-0.379351,-0.384721
4,truthfulqa_00004_t0.1_ans00,truthfulqa,-44.235367,17.476067,3.255803,-17.599913,8.564972,-6.400651,-15.197990,3.964259,...,-0.292882,-0.094813,-0.288577,-0.113414,0.057697,0.639345,0.199955,0.200291,-0.392096,-0.222924


In [12]:
import torch

torch.save(
    {
        "X": torch.tensor(
            df_pca.filter(like="PC").values,
            dtype=torch.float32
        ),
        "id": df_pca["id"].tolist(),
        "dataset": df_pca["dataset"].tolist()
    },
    "../data/processed/df_pca.pt"
)